# VisionDocPhi-3.5 — Resume Expansion (Kaggle T4)

**Phases:** Baseline / Adaptive V2 → Eval harness → Vector index → QLoRA → Quant bench → Guardrails smoke.

Follow step-by-step Cursor prompts + tests in `docs/RESUME_EXPANSION_PLAYBOOK.md`.

In [ ]:
import os
import sys

PROJECT_NAME = "VisionDocPhi-3.5"
GITHUB_REPO = "https://github.com/mokshu7k/VisionDocPhi-3.5.git"

if os.path.exists("/kaggle/working"):
    PROJECT_PATH = os.path.join("/kaggle/working", PROJECT_NAME)
else:
    PROJECT_PATH = os.path.join("/content", PROJECT_NAME)

if not os.path.exists(PROJECT_PATH):
    print("Cloning repository...")
    parent = os.path.dirname(PROJECT_PATH)
    os.makedirs(parent, exist_ok=True)
    os.chdir(parent)
    os.system(f"git clone {GITHUB_REPO} {PROJECT_NAME}")

os.chdir(PROJECT_PATH)
sys.path.insert(0, PROJECT_PATH)
os.environ["DEVICE"] = "cuda"
os.environ["USE_8BIT_QUANTIZATION"] = "false"
os.environ["USE_CHROMA"] = "true"
print(f"Project path: {PROJECT_PATH}")

if os.path.exists(os.path.join(PROJECT_PATH, ".git")):
    os.system("git pull --rebase || git pull")


## Install dependencies

In [ ]:
!pip install -q sentence-transformers opencv-python-headless peft accelerate chromadb
# Optional for QLoRA 4-bit base (may fail on some Kaggle CUDA images — training falls back):
# !pip install -q bitsandbytes


## GPU check

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")


## Import / settings smoke

In [ ]:
from config.settings import MODEL_NAME, EVAL_SUBSET_FILE, ADAPTER_PATH, CHROMA_DIR, HARNESS_DIR
print("MODEL_NAME:", MODEL_NAME)
print("EVAL_SUBSET_FILE:", EVAL_SUBSET_FILE)
print("ADAPTER_PATH:", ADAPTER_PATH)
print("CHROMA_DIR:", CHROMA_DIR)
print("HARNESS_DIR:", HARNESS_DIR)


## Build eval subset (200 samples)

In [ ]:
from pathlib import Path
subset_path = Path("data/outputs/eval_subset_200.json")
if subset_path.exists():
    print(f"Subset exists: {subset_path}")
else:
    !python scripts/build_eval_subset.py


## Optional: 2-sample smoke (set RUN_SMOKE=True)

In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    !python scripts/chunked_evaluation.py --mode baseline --subset data/outputs/eval_subset_200.json --chunk-size 2 --no-resume
else:
    print("Skipping smoke. Set RUN_SMOKE=True to run a 2-sample chunk.")


## Baseline evaluation (`vision_only`)

In [ ]:
!python scripts/chunked_evaluation.py --mode baseline --subset data/outputs/eval_subset_200.json --chunk-size 20 --resume


## Adaptive evaluation (`ocr_adaptive` v2)

In [ ]:
!python scripts/chunked_evaluation.py --mode ocr_adaptive --version v2 --subset data/outputs/eval_subset_200.json --chunk-size 20 --resume


## Compare baseline vs adaptive + harness report

In [ ]:
!python scripts/compare_baselines.py --adaptive-version v2
!python scripts/run_harness.py --compare
!python scripts/run_harness.py --dry-run


## Vector OCR index smoke (Chroma / numpy fallback)

In [ ]:
!python scripts/build_ocr_index.py --smoke


## QLoRA smoke train (CPU stub locally; real weights on GPU)

In [ ]:
# For real QLoRA NF4 base on T4 you may enable:
# import os; os.environ["USE_8BIT_QUANTIZATION"] = "true"
!python scripts/train_qlora.py --smoke


## QLoRA full train (requires adaptive merged results + GPU)

In [ ]:
RUN_QLORA_FULL = False
if RUN_QLORA_FULL:
    !python scripts/train_qlora.py --merged data/outputs/ocr_adaptive_200_v2/results_merged.json
else:
    print("Set RUN_QLORA_FULL=True after adaptive merge exists (playbook P3).")


## Eval with adapter (set ADAPTER_PATH; version qlora)

In [ ]:
RUN_QLORA_EVAL = False
if RUN_QLORA_EVAL:
    import os
    os.environ["ADAPTER_PATH"] = "data/outputs/adapters/phi35_qlora"
    !python scripts/chunked_evaluation.py --mode ocr_adaptive --version qlora --subset data/outputs/eval_subset_200.json --chunk-size 20 --resume
    !python scripts/run_harness.py --compare --qlora
else:
    print("Set RUN_QLORA_EVAL=True after adapter training (playbook P3).")


## Quantization bench (fp16 vs NF4)

In [ ]:
!python scripts/quant_bench.py --formats fp16,nf4
from pathlib import Path
p = Path("data/outputs/quant_bench.md")
print(p.read_text() if p.exists() else "quant_bench.md not written yet")


## Guardrails / tools / unit tests smoke

In [ ]:
!pytest tests/test_guardrails.py tests/test_expansion_smoke.py -q || true
!python scripts/run_tools_demo.py --smoke || true
print("If modules missing, finish playbook P0 then P4 first.")


## Verify resume-expansion checklist

In [ ]:
!python scripts/verify_resume_expansion.py || true


## Git backup outputs (optional)

In [ ]:
import os
os.system("git config user.email 'mokshu7k@users.noreply.github.com'")
os.system("git config user.name 'mokshu7k'")
os.system("git add data/outputs/eval_subset_200.json data/outputs/harness data/outputs/quant_bench.md data/outputs/quant_bench.json || true")
os.system("git add data/outputs/baseline_200 data/outputs/ocr_adaptive_200_v2 data/outputs/comparisons || true")
os.system("git add data/outputs/adapters || true")
os.system("git status")
os.system("git commit -m 'Kaggle resume-expansion outputs' || true")
# os.system("git push origin main")


## Notes

- Frozen eval: `USE_8BIT_QUANTIZATION=false` (float16) — stable on Kaggle T4
- QLoRA NF4: enable `USE_8BIT_QUANTIZATION=true` only in train cells; fall back if bnb fails
- Chunk size 20 + `--resume` across sessions
- **Playbook (prompts + tests):** `docs/RESUME_EXPANSION_PLAYBOOK.md`
- Archive legacy `chunks_val/` — `docs/ARCHIVE_LEGACY_CHUNKS.md`
- Attach SP-DocVQA images + Azure OCR datasets before full 200-sample runs
